In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
from pathlib import Path
output_dir = Path("C:/Users/Ayush/Documents/Research Papers")

# Model assumptions
RISK_FREE_RATE = 0.065
DIVIDEND_YIELD = 0.0
TRADING_DAYS = 252
TWO_YEAR_RETURNS = 504

# Stock ticker mapping
TICKER_MAP = {
    "ADANIENT": "ADANIENT.NS",
    "HINDUNILVR": "HINDUNILVR.NS",
}

print("Setup complete.")

Setup complete.


In [5]:
def black_scholes_call(S, K, T, r, q, sigma):
    if T <= 0:
        return max(S - K, 0.0)
    if sigma <= 0:
        return max(S * np.exp(-q*T) - K * np.exp(-r*T), 0.0)

    d1 = (
        np.log(S / K) + (r - q + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2 = d1 - sigma * np.sqrt(T)

    return (
        S * np.exp(-q*T) * norm.cdf(d1)
        - K * np.exp(-r*T) * norm.cdf(d2)
    )


def bachelier_call(S, K, T, r, q, sigma_n):
    if T <= 0:
        return max(S - K, 0.0)

    forward = S * np.exp((r - q) * T)
    scale = sigma_n * np.sqrt(T)

    if scale <= 0:
        return np.exp(-r*T) * max(forward - K, 0.0)

    d = (forward - K) / scale

    return np.exp(-r*T) * (
        (forward - K) * norm.cdf(d)
        + scale * norm.pdf(d)
    )


def analyse_options_csv(
    csv_path,
    max_strikes_per_expiry=3,
    history_start="2021-01-01"
):
    # 1. Load and clean the NSE CSV
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    required = [
        "Symbol", "Date", "Expiry", "Option type",
        "Strike Price", "Close", "No. of contracts",
        "Underlying Value"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df["Expiry"] = pd.to_datetime(df["Expiry"], dayfirst=True, errors="coerce")

    numeric_cols = [
        "Strike Price", "Close", "No. of contracts",
        "Underlying Value"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Symbol"] = df["Symbol"].astype(str).str.strip().str.upper()
    df["Option type"] = df["Option type"].astype(str).str.strip().str.upper()

    symbols = df["Symbol"].dropna().unique()

    if len(symbols) != 1:
        raise ValueError(
            f"Expected one stock per CSV, found: {symbols.tolist()}"
        )

    symbol = symbols[0]

    if symbol not in TICKER_MAP:
        raise ValueError(
            f"Unknown symbol '{symbol}'. Add it to TICKER_MAP."
        )

    ticker = TICKER_MAP[symbol]

    # Keep valid call options with a usable market price and positive trading
    df = df[
        (df["Option type"] == "CE") &
        (df["Close"] >= 0) &
        (df["No. of contracts"] > 0) &
        df["Date"].notna() &
        df["Expiry"].notna() &
        (df["Strike Price"] > 0) &
        (df["Underlying Value"] > 0)
    ].copy()

    if df.empty:
        raise ValueError("No valid traded call options found in this CSV.")

    # 2. Download stock history covering all valuation dates
    start = pd.Timestamp(history_start)
    last_date = df["Date"].max()

    stock = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=(last_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False
    )

    if stock.empty:
        raise ValueError(f"No stock history downloaded for {ticker}.")

    # Handle yfinance's MultiIndex columns
    close = stock["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.dropna().sort_index()

    # 3. Calculate historical volatility using data up to each valuation date
    results = []

    for valuation_date, day_data in df.groupby("Date"):
        valuation_date = pd.Timestamp(valuation_date).normalize()

        historical_close = close.loc[close.index <= valuation_date]

        if len(historical_close) < 61:
            print(f"Skipping {valuation_date.date()}: insufficient stock history.")
            continue

        log_returns = np.log(historical_close / historical_close.shift(1)).dropna()
        price_changes = historical_close.diff().dropna()

        # Full available history
        bs_vol_full = log_returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
        bach_vol_full = price_changes.std(ddof=1) * np.sqrt(TRADING_DAYS)

        # Most recent 60 daily returns / price changes
        bs_vol_60 = log_returns.tail(60).std(ddof=1) * np.sqrt(TRADING_DAYS)
        bach_vol_60 = price_changes.tail(60).std(ddof=1) * np.sqrt(TRADING_DAYS)

        # Exclude expired options
        day_data = day_data[day_data["Expiry"] > valuation_date].copy()

        if day_data.empty:
            continue

        # Use the nearest strikes to spot for each expiry
        spot = float(day_data["Underlying Value"].median())
        day_data["Strike Distance"] = abs(day_data["Strike Price"] - spot)

        day_data = (
            day_data.sort_values("Strike Distance")
            .groupby("Expiry", group_keys=False)
            .head(max_strikes_per_expiry)
        )

        for _, row in day_data.iterrows():
            S = float(row["Underlying Value"])
            K = float(row["Strike Price"])
            market_price = float(row["Close"])
            expiry = pd.Timestamp(row["Expiry"]).normalize()

            T = (expiry - valuation_date).days / 365.0

            if T <= 0:
                continue

            for window, bs_vol, bach_vol in [
                ("Full period", bs_vol_full, bach_vol_full),
                ("60 days", bs_vol_60, bach_vol_60)
            ]:
                bs_price = black_scholes_call(
                    S, K, T, RISK_FREE_RATE, DIVIDEND_YIELD, bs_vol
                )

                bach_price = bachelier_call(
                    S, K, T, RISK_FREE_RATE, DIVIDEND_YIELD, bach_vol
                )

                results.append({
                    "Symbol": symbol,
                    "Valuation Date": valuation_date,
                    "Expiry": expiry,
                    "Days to Expiry": (expiry - valuation_date).days,
                    "Spot": S,
                    "Strike": K,
                    "Market Price": market_price,
                    "Volatility Window": window,
                    "BS Volatility": bs_vol,
                    "Bachelier Volatility": bach_vol,
                    "BS Price": bs_price,
                    "Bachelier Price": bach_price,
                    "BS Absolute Error": abs(bs_price - market_price),
                    "Bachelier Absolute Error": abs(bach_price - market_price),
                    "Contracts": row["No. of contracts"]
                })

    results_df = pd.DataFrame(results)

    if results_df.empty:
        raise ValueError(
            "No results generated. Check the dates and stock-history coverage."
        )

    # 4. Overall summary by volatility window
    summary = (
        results_df.groupby("Volatility Window")
        .agg(
            BS_MAE=("BS Absolute Error", "mean"),
            Bachelier_MAE=("Bachelier Absolute Error", "mean"),
            Options_Analysed=("Strike", "size")
        )
        .round(2)
    )

    # 5. Summary by valuation date and volatility window
    daily_summary = (
        results_df.groupby(["Valuation Date", "Volatility Window"])
        .agg(
            BS_MAE=("BS Absolute Error", "mean"),
            Bachelier_MAE=("Bachelier Absolute Error", "mean"),
            Options_Analysed=("Strike", "size")
        )
        .round(2)
        .reset_index()
    )

    print(f"\nStock: {symbol} ({ticker})")
    print(f"Options file: {Path(csv_path).name}")
    print("\nOVERALL RESULTS — lower MAE means closer to market prices")
    display(summary)

    print("\nRESULTS BY DATE")
    display(daily_summary)

    print("\nINDIVIDUAL OPTION RESULTS")
    display(results_df.round(3))

    print(f"\nSaved detailed results and summaries to: {output_dir}")

    return results_df, summary, daily_summary

ADANI (MORE VOLATILE)

In [6]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_ADANIENT_CE_01-Dec-2025_TO_31-Dec-2025.csv"
)


Stock: ADANIENT (ADANIENT.NS)
Options file: OPTSTK_ADANIENT_CE_01-Dec-2025_TO_31-Dec-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,3.72,4.95,133
Full period,58.63,60.89,133



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-12-01,60 days,4.42,3.62,6
1,2025-12-01,Full period,67.15,68.36,6
2,2025-12-02,60 days,2.83,3.86,6
3,2025-12-02,Full period,67.27,69.52,6
4,2025-12-03,60 days,4.12,5.91,7
5,2025-12-03,Full period,66.23,71.02,7
6,2025-12-04,60 days,2.15,8.06,6
7,2025-12-04,Full period,67.67,70.51,6
8,2025-12-05,60 days,7.62,13.14,6
9,2025-12-05,Full period,73.28,74.21,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2260.0,77.40,Full period,0.513,1170.929,136.795,137.927,59.395,60.527,764.0
1,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2260.0,77.40,60 days,0.275,670.523,76.713,82.031,0.687,4.631,764.0
2,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2268.6,73.50,Full period,0.513,1170.929,132.666,133.552,59.166,60.052,182.0
3,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2268.6,73.50,60 days,0.275,670.523,72.385,77.583,1.115,4.083,182.0
4,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2249.2,82.50,Full period,0.513,1170.929,142.108,143.548,59.608,61.048,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2220.0,74.10,60 days,0.240,561.052,74.302,76.964,0.202,2.864,1494.0
262,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2260.0,53.45,Full period,0.509,1162.613,119.166,120.870,65.716,67.420,1147.0
263,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2260.0,53.45,60 days,0.240,561.052,53.777,55.972,0.327,2.522,1147.0
264,ADANIENT,2025-12-31,2026-02-24,55,2239.7,2200.0,115.00,Full period,0.509,1162.613,206.094,210.527,91.094,95.527,22.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [7]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_ADANIENT_CE_01-Nov-2025_TO_30-Nov-2025.csv"
)


Stock: ADANIENT (ADANIENT.NS)
Options file: OPTSTK_ADANIENT_CE_01-Nov-2025_TO_30-Nov-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,10.10,10.05,120
Full period,59.05,52.10,120



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-11-03,60 days,25.23,27.01,6
1,2025-11-03,Full period,54.64,42.58,6
2,2025-11-04,60 days,25.30,25.96,7
3,2025-11-04,Full period,57.43,47.07,7
4,2025-11-06,60 days,12.85,10.20,6
5,2025-11-06,Full period,55.54,52.51,6
6,2025-11-07,60 days,13.20,11.55,7
7,2025-11-07,Full period,62.13,56.11,7
8,2025-11-10,60 days,11.47,9.27,6
9,2025-11-10,Full period,56.64,51.81,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,ADANIENT,2025-11-03,2025-11-25,22,2467.0,2460.0,92.90,Full period,0.515,1175.946,132.374,123.227,39.474,30.327,1002.0
1,ADANIENT,2025-11-03,2025-11-25,22,2467.0,2460.0,92.90,60 days,0.257,621.609,70.588,69.316,22.312,23.584,1002.0
2,ADANIENT,2025-11-03,2025-11-25,22,2467.0,2480.0,82.40,Full period,0.515,1175.946,122.745,113.082,40.345,30.682,1959.0
3,ADANIENT,2025-11-03,2025-11-25,22,2467.0,2480.0,82.40,60 days,0.257,621.609,60.538,59.007,21.862,23.393,1959.0
4,ADANIENT,2025-11-03,2025-12-30,57,2467.0,2480.0,140.00,Full period,0.515,1175.946,205.468,189.605,65.468,49.605,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,ADANIENT,2025-11-28,2026-01-27,60,2280.2,2300.0,106.05,60 days,0.275,670.174,103.486,109.586,2.564,3.536,110.0
236,ADANIENT,2025-11-28,2026-01-27,60,2280.2,2320.0,115.00,Full period,0.513,1171.377,181.910,179.980,66.910,64.980,2.0
237,ADANIENT,2025-11-28,2026-01-27,60,2280.2,2320.0,115.00,60 days,0.275,670.174,94.194,99.846,20.806,15.154,2.0
238,ADANIENT,2025-11-28,2026-01-27,60,2280.2,2340.0,112.70,Full period,0.513,1171.377,173.246,170.507,60.546,57.807,7.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [8]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_ADANIENT_CE_01-Oct-2025_TO_31-Oct-2025.csv"
)


Stock: ADANIENT (ADANIENT.NS)
Options file: OPTSTK_ADANIENT_CE_01-Oct-2025_TO_31-Oct-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,11.82,15.49,129
Full period,57.60,42.91,129



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-10-01,60 days,10.23,16.72,6
1,2025-10-01,Full period,69.40,47.96,6
2,2025-10-03,60 days,12.01,18.10,6
3,2025-10-03,Full period,65.25,44.78,6
4,2025-10-06,60 days,14.98,20.39,6
5,2025-10-06,Full period,58.55,39.71,6
6,2025-10-07,60 days,15.49,20.16,6
7,2025-10-07,Full period,55.93,38.55,6
8,2025-10-08,60 days,22.28,26.00,6
9,2025-10-08,Full period,48.02,38.28,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,ADANIENT,2025-10-01,2025-10-28,27,2592.7,2600.0,84.50,Full period,0.519,1185.501,148.347,130.617,63.847,46.117,11569.0
1,ADANIENT,2025-10-01,2025-10-28,27,2592.7,2600.0,84.50,60 days,0.281,681.716,81.632,76.229,2.868,8.271,11569.0
2,ADANIENT,2025-10-01,2025-11-25,55,2592.7,2600.0,140.95,Full period,0.519,1185.501,216.467,190.963,75.517,50.013,61.0
3,ADANIENT,2025-10-01,2025-11-25,55,2592.7,2600.0,140.95,60 days,0.281,681.716,121.722,113.811,19.228,27.139,61.0
4,ADANIENT,2025-10-01,2025-10-28,27,2592.7,2580.0,93.50,Full period,0.519,1185.501,158.068,140.943,64.568,47.443,420.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,ADANIENT,2025-10-31,2025-12-30,60,2481.0,2500.0,130.50,60 days,0.259,625.539,107.538,103.935,22.962,26.565,56.0
254,ADANIENT,2025-10-31,2025-11-25,25,2481.0,2460.0,104.10,Full period,0.515,1176.417,149.020,138.908,44.920,34.808,258.0
255,ADANIENT,2025-10-31,2025-11-25,25,2481.0,2460.0,104.10,60 days,0.259,625.539,83.809,82.229,20.291,21.871,258.0
256,ADANIENT,2025-10-31,2025-12-30,60,2481.0,2560.0,114.65,Full period,0.515,1176.417,183.568,163.498,68.918,48.848,1.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [9]:
# Analyse Adani Enterprises: October, November and December 2025
adani_files = [
    "OPTSTK_ADANIENT_CE_01-Oct-2025_TO_31-Oct-2025.csv",
    "OPTSTK_ADANIENT_CE_01-Nov-2025_TO_30-Nov-2025.csv",
    "OPTSTK_ADANIENT_CE_01-Dec-2025_TO_31-Dec-2025.csv",
]
adani_results_list = []
adani_summaries_list = []
adani_daily_list = []
for file in adani_files:
    results, summary, daily_summary = analyse_options_csv(file)
    adani_results_list.append(results)
    adani_summaries_list.append(summary.reset_index())
    adani_daily_list.append(daily_summary)
adani_results_all = pd.concat(adani_results_list, ignore_index=True)
adani_summary_all = pd.concat(adani_summaries_list, ignore_index=True)
adani_daily_all = pd.concat(adani_daily_list, ignore_index=True)


Stock: ADANIENT (ADANIENT.NS)
Options file: OPTSTK_ADANIENT_CE_01-Oct-2025_TO_31-Oct-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,11.82,15.49,129
Full period,57.60,42.91,129



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-10-01,60 days,10.23,16.72,6
1,2025-10-01,Full period,69.40,47.96,6
2,2025-10-03,60 days,12.01,18.10,6
3,2025-10-03,Full period,65.25,44.78,6
4,2025-10-06,60 days,14.98,20.39,6
5,2025-10-06,Full period,58.55,39.71,6
6,2025-10-07,60 days,15.49,20.16,6
7,2025-10-07,Full period,55.93,38.55,6
8,2025-10-08,60 days,22.28,26.00,6
9,2025-10-08,Full period,48.02,38.28,6



INDIVIDUAL OPTION RESULTS


...

,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-12-01,60 days,4.42,3.62,6
1,2025-12-01,Full period,67.15,68.36,6
2,2025-12-02,60 days,2.83,3.86,6
3,2025-12-02,Full period,67.27,69.52,6
4,2025-12-03,60 days,4.12,5.91,7
5,2025-12-03,Full period,66.23,71.02,7
6,2025-12-04,60 days,2.15,8.06,6
7,2025-12-04,Full period,67.67,70.51,6
8,2025-12-05,60 days,7.62,13.14,6
9,2025-12-05,Full period,73.28,74.21,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2260.0,77.40,Full period,0.513,1170.929,136.795,137.927,59.395,60.527,764.0
1,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2260.0,77.40,60 days,0.275,670.523,76.713,82.031,0.687,4.631,764.0
2,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2268.6,73.50,Full period,0.513,1170.929,132.666,133.552,59.166,60.052,182.0
3,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2268.6,73.50,60 days,0.275,670.523,72.385,77.583,1.115,4.083,182.0
4,ADANIENT,2025-12-01,2025-12-30,29,2262.0,2249.2,82.50,Full period,0.513,1170.929,142.108,143.548,59.608,61.048,60.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2220.0,74.10,60 days,0.240,561.052,74.302,76.964,0.202,2.864,1494.0
262,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2260.0,53.45,Full period,0.509,1162.613,119.166,120.870,65.716,67.420,1147.0
263,ADANIENT,2025-12-31,2026-01-27,27,2239.7,2260.0,53.45,60 days,0.240,561.052,53.777,55.972,0.327,2.522,1147.0
264,ADANIENT,2025-12-31,2026-02-24,55,2239.7,2200.0,115.00,Full period,0.509,1162.613,206.094,210.527,91.094,95.527,22.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [10]:
# Combined Adani comparison
adani_comparison = (
    adani_results_all
    .groupby("Volatility Window")
    .agg(
        BS_MAE=("BS Absolute Error", "mean"),
        Bachelier_MAE=("Bachelier Absolute Error", "mean"),
        Options_Analysed=("Strike", "count")
    )
    .round(2)
)
display(adani_comparison)

,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,8.46,10.11,382
Full period,58.41,52.06,382


HUL (LESS VOLATILE)

In [11]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_HINDUNILVR_CE_01-Dec-2025_TO_31-Dec-2025.csv"
)


Stock: HINDUNILVR (HINDUNILVR.NS)
Options file: OPTSTK_HINDUNILVR_CE_01-Dec-2025_TO_31-Dec-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,10.84,11.95,133
Full period,19.58,18.35,133



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-12-01,60 days,28.01,28.38,6
1,2025-12-01,Full period,47.55,42.98,6
2,2025-12-02,60 days,33.23,33.06,7
3,2025-12-02,Full period,54.03,48.35,7
4,2025-12-03,60 days,35.40,35.66,7
5,2025-12-03,Full period,55.64,50.80,7
6,2025-12-04,60 days,43.64,43.73,9
7,2025-12-04,Full period,65.21,59.81,9
8,2025-12-05,60 days,5.89,6.01,6
9,2025-12-05,Full period,8.66,7.96,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2460.0,27.85,Full period,0.201,461.034,64.501,60.617,36.651,32.767,6200.0
1,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2460.0,27.85,60 days,0.139,346.227,47.560,47.923,19.710,20.073,6200.0
2,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2480.0,17.60,Full period,0.201,461.034,54.366,50.225,36.766,32.625,16033.0
3,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2480.0,17.60,60 days,0.139,346.227,37.172,37.386,19.572,19.786,16033.0
4,HINDUNILVR,2025-12-01,2026-01-27,57,2464.5,2440.0,40.20,Full period,0.201,461.034,104.326,99.175,64.126,58.975,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2320.0,41.95,60 days,0.172,413.019,46.675,48.201,4.725,6.251,2713.0
262,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2300.0,53.15,Full period,0.201,461.953,64.885,64.502,11.735,11.352,5204.0
263,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2300.0,53.15,60 days,0.172,413.019,57.689,59.353,4.539,6.203,5204.0
264,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2340.0,31.25,Full period,0.201,461.953,44.532,43.710,13.282,12.460,2718.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [12]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_HINDUNILVR_CE_01-Nov-2025_TO_30-Nov-2025.csv"
)


Stock: HINDUNILVR (HINDUNILVR.NS)
Options file: OPTSTK_HINDUNILVR_CE_01-Nov-2025_TO_30-Nov-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,11.25,12.10,132
Full period,22.84,19.28,132



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-11-03,60 days,10.30,11.32,6
1,2025-11-03,Full period,18.60,13.89,6
2,2025-11-04,60 days,12.23,13.70,6
3,2025-11-04,Full period,20.10,16.09,6
4,2025-11-06,60 days,11.28,12.74,7
5,2025-11-06,Full period,19.64,15.35,7
6,2025-11-07,60 days,3.30,4.31,6
7,2025-11-07,Full period,8.99,6.14,6
8,2025-11-10,60 days,1.99,2.37,6
9,2025-11-10,Full period,7.01,4.09,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,HINDUNILVR,2025-11-03,2025-11-25,22,2460.0,2460.0,38.35,Full period,0.202,463.645,53.566,50.206,15.216,11.856,2463.0
1,HINDUNILVR,2025-11-03,2025-11-25,22,2460.0,2460.0,38.35,60 days,0.175,443.119,47.172,48.211,8.822,9.861,2463.0
2,HINDUNILVR,2025-11-03,2025-12-30,57,2460.0,2460.0,67.00,Full period,0.202,463.645,91.031,85.457,24.031,18.457,14.0
3,HINDUNILVR,2025-11-03,2025-12-30,57,2460.0,2460.0,67.00,60 days,0.175,443.119,80.835,82.285,13.835,15.285,14.0
4,HINDUNILVR,2025-11-03,2025-11-25,22,2460.0,2440.0,48.05,Full period,0.202,463.645,64.605,61.530,16.555,13.480,983.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,HINDUNILVR,2025-11-28,2026-01-27,60,2466.6,2520.0,10.40,60 days,0.139,346.725,43.487,43.190,33.087,32.790,520.0
260,HINDUNILVR,2025-11-28,2026-01-27,60,2466.6,2400.0,77.50,Full period,0.201,461.223,133.165,128.825,55.665,51.325,249.0
261,HINDUNILVR,2025-11-28,2026-01-27,60,2466.6,2400.0,77.50,60 days,0.139,346.725,112.450,113.281,34.950,35.781,249.0
262,HINDUNILVR,2025-11-28,2026-02-24,88,2466.6,2400.0,99.00,Full period,0.201,461.223,155.902,150.392,56.902,51.392,1.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [13]:
results, summary, daily_summary = analyse_options_csv(
    "OPTSTK_HINDUNILVR_CE_01-Oct-2025_TO_31-Oct-2025.csv"
)


Stock: HINDUNILVR (HINDUNILVR.NS)
Options file: OPTSTK_HINDUNILVR_CE_01-Oct-2025_TO_31-Oct-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,8.34,7.96,133
Full period,12.43,8.04,133



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-10-01,60 days,14.93,13.31,6
1,2025-10-01,Full period,16.32,9.97,6
2,2025-10-03,60 days,13.51,11.57,6
3,2025-10-03,Full period,15.14,8.28,6
4,2025-10-06,60 days,5.90,5.17,6
5,2025-10-06,Full period,7.83,4.82,6
6,2025-10-07,60 days,8.09,7.27,7
7,2025-10-07,Full period,9.56,6.39,7
8,2025-10-08,60 days,4.76,4.46,6
9,2025-10-08,Full period,10.13,5.11,6



INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,HINDUNILVR,2025-10-01,2025-10-28,27,2536.2,2540.0,51.45,Full period,0.203,464.471,59.983,54.458,8.533,3.008,1479.0
1,HINDUNILVR,2025-10-01,2025-10-28,27,2536.2,2540.0,51.45,60 days,0.198,492.498,58.746,57.479,7.296,6.029,1479.0
2,HINDUNILVR,2025-10-01,2025-10-28,27,2536.2,2520.0,62.30,Full period,0.203,464.471,70.759,65.563,8.459,3.263,1155.0
3,HINDUNILVR,2025-10-01,2025-10-28,27,2536.2,2520.0,62.30,60 days,0.198,492.498,69.550,68.518,7.250,6.218,1155.0
4,HINDUNILVR,2025-10-01,2025-10-28,27,2536.2,2560.0,41.60,Full period,0.203,464.471,50.338,44.605,8.738,3.005,954.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,HINDUNILVR,2025-10-31,2025-11-25,25,2465.5,2440.0,55.70,60 days,0.176,443.313,65.339,66.511,9.639,10.811,261.0
262,HINDUNILVR,2025-10-31,2025-12-30,60,2465.5,2500.0,54.80,Full period,0.202,463.832,76.850,70.329,22.050,15.529,195.0
263,HINDUNILVR,2025-10-31,2025-12-30,60,2465.5,2500.0,54.80,60 days,0.176,443.313,66.204,67.049,11.404,12.249,195.0
264,HINDUNILVR,2025-10-31,2025-12-30,60,2465.5,2400.0,107.50,Full period,0.202,463.832,132.816,128.437,25.316,20.937,77.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


In [14]:
# Store results from each monthly run
all_results = []
all_summaries = []
all_daily_summaries = []

files_to_analyse = [
    "OPTSTK_HINDUNILVR_CE_01-Oct-2025_TO_31-Oct-2025.csv",
    "OPTSTK_HINDUNILVR_CE_01-Nov-2025_TO_30-Nov-2025.csv",
    "OPTSTK_HINDUNILVR_CE_01-Dec-2025_TO_31-Dec-2025.csv",
]

for file in files_to_analyse:
    results, summary, daily_summary = analyse_options_csv(file)

    all_results.append(results)
    all_summaries.append(summary.reset_index())
    all_daily_summaries.append(daily_summary)

# Combine all three months in memory
hul_results_all = pd.concat(all_results, ignore_index=True)
hul_summary_all = pd.concat(all_summaries, ignore_index=True)
hul_daily_all = pd.concat(all_daily_summaries, ignore_index=True)

# Display the combined results
display(hul_summary_all)


Stock: HINDUNILVR (HINDUNILVR.NS)
Options file: OPTSTK_HINDUNILVR_CE_01-Oct-2025_TO_31-Oct-2025.csv

OVERALL RESULTS — lower MAE means closer to market prices


,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,8.34,7.96,133
Full period,12.43,8.04,133



RESULTS BY DATE


,Valuation Date,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,2025-10-01,60 days,14.93,13.31,6
1,2025-10-01,Full period,16.32,9.97,6
2,2025-10-03,60 days,13.51,11.57,6
3,2025-10-03,Full period,15.14,8.28,6
4,2025-10-06,60 days,5.90,5.17,6
5,2025-10-06,Full period,7.83,4.82,6
6,2025-10-07,60 days,8.09,7.27,7
7,2025-10-07,Full period,9.56,6.39,7
8,2025-10-08,60 days,4.76,4.46,6
9,2025-10-08,Full period,10.13,5.11,6



INDIVIDUAL OPTION RESULTS


...


INDIVIDUAL OPTION RESULTS


C:\Users\Ayush\AppData\Local\Temp\ipykernel_12984\1121500276.py:242: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(results_df.round(3))


,Symbol,Valuation Date,Expiry,Days to Expiry,Spot,Strike,Market Price,Volatility Window,BS Volatility,Bachelier Volatility,BS Price,Bachelier Price,BS Absolute Error,Bachelier Absolute Error,Contracts
0,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2460.0,27.85,Full period,0.201,461.034,64.501,60.617,36.651,32.767,6200.0
1,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2460.0,27.85,60 days,0.139,346.227,47.560,47.923,19.710,20.073,6200.0
2,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2480.0,17.60,Full period,0.201,461.034,54.366,50.225,36.766,32.625,16033.0
3,HINDUNILVR,2025-12-01,2025-12-30,29,2464.5,2480.0,17.60,60 days,0.139,346.227,37.172,37.386,19.572,19.786,16033.0
4,HINDUNILVR,2025-12-01,2026-01-27,57,2464.5,2440.0,40.20,Full period,0.201,461.034,104.326,99.175,64.126,58.975,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2320.0,41.95,60 days,0.172,413.019,46.675,48.201,4.725,6.251,2713.0
262,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2300.0,53.15,Full period,0.201,461.953,64.885,64.502,11.735,11.352,5204.0
263,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2300.0,53.15,60 days,0.172,413.019,57.689,59.353,4.539,6.203,5204.0
264,HINDUNILVR,2025-12-31,2026-01-27,27,2315.9,2340.0,31.25,Full period,0.201,461.953,44.532,43.710,13.282,12.460,2718.0



Saved detailed results and summaries to: C:\Users\Ayush\Documents\Research Papers


,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,60 days,8.34,7.96,133
1,Full period,12.43,8.04,133
2,60 days,11.25,12.10,132
3,Full period,22.84,19.28,132
4,60 days,10.84,11.95,133
5,Full period,19.58,18.35,133


In [15]:
# Compare HUL across October, November and December

hul_comparison = (
    hul_results_all
    .groupby("Volatility Window")
    .agg(
        BS_MAE=("BS Absolute Error", "mean"),
        Bachelier_MAE=("Bachelier Absolute Error", "mean"),
        Options_Analysed=("Strike", "count")
    )
    .round(2)
)

display(hul_comparison)

,BS_MAE,Bachelier_MAE,Options_Analysed
Volatility Window,,,
60 days,10.14,10.67,398
Full period,18.27,15.22,398


In [16]:
# Combine Adani and HUL results in memory

adani_tagged = adani_results_all.copy()
hul_tagged = hul_results_all.copy()

adani_tagged["Stock"] = "ADANIENT"
hul_tagged["Stock"] = "HINDUNILVR"

combined_results = pd.concat(
    [adani_tagged, hul_tagged],
    ignore_index=True
)

# Compare model errors by stock and volatility window
stock_comparison = (
    combined_results
    .groupby(["Stock", "Volatility Window"])
    .agg(
        BS_MAE=("BS Absolute Error", "mean"),
        Bachelier_MAE=("Bachelier Absolute Error", "mean"),
        Options_Analysed=("Strike", "count")
    )
    .round(2)
    .reset_index()
)

display(stock_comparison)

,Stock,Volatility Window,BS_MAE,Bachelier_MAE,Options_Analysed
0,ADANIENT,60 days,8.46,10.11,382
1,ADANIENT,Full period,58.41,52.06,382
2,HINDUNILVR,60 days,10.14,10.67,398
3,HINDUNILVR,Full period,18.27,15.22,398


## PIPELINE FOR 2 YEARS

In [17]:
def analyse_options_csv(
    csv_path,
    stock_label,
    ticker,
    max_strikes_per_expiry=3
):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    required = [
        "Date", "Expiry", "Strike Price", "Close",
        "No. of contracts", "Underlying Value", "Option type"
    ]
    
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path}: missing columns {missing}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    df["Expiry"] = pd.to_datetime(df["Expiry"], errors="coerce").dt.normalize()

    for col in ["Strike Price", "Close", "No. of contracts", "Underlying Value"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Evaluation period: January 2023 through December 2025
    df = df[
        df["Date"].between("2023-01-01", "2025-12-31")
        & (df["Option type"].astype(str).str.strip().str.upper() == "CE")
        & (df["Close"] >= 0)
        & (df["No. of contracts"] > 0)
        & (df["Strike Price"] > 0)
        & (df["Underlying Value"] > 0)
        & df["Expiry"].notna()
    ].copy()

    if df.empty:
        return pd.DataFrame()

    # Download adjusted prices to estimate historical volatility.
    # End date is exclusive, so include the last day of 2025.
    stock = yf.download(
        ticker,
        start="2020-12-01",
        end="2026-01-03",
        auto_adjust=True,
        progress=False
    )

    if stock.empty:
        raise ValueError(f"No Yahoo Finance data returned for {ticker}")

    close = stock["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    close = close.dropna()
    close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
    close = close[~close.index.duplicated(keep="last")].sort_index()

    log_returns = np.log(close / close.shift(1)).dropna()
    price_changes = close.diff().dropna()

    records = []

    # Estimate volatility separately on each option valuation date.
    for valuation_date, day in df.groupby("Date"):
        past_returns = log_returns.loc[log_returns.index <= valuation_date]
        past_changes = price_changes.loc[price_changes.index <= valuation_date]

        vol_lr_60 = past_returns.tail(60).std(ddof=1) * np.sqrt(TRADING_DAYS)
        vol_lr_2y = past_returns.tail(TWO_YEAR_RETURNS).std(ddof=1) * np.sqrt(TRADING_DAYS)

        vol_change_60 = past_changes.tail(60).std(ddof=1) * np.sqrt(TRADING_DAYS)
        vol_change_2y = past_changes.tail(TWO_YEAR_RETURNS).std(ddof=1) * np.sqrt(TRADING_DAYS)

        # Skip dates without sufficient data for both windows.
        if (
            len(past_returns.tail(60)) < 60
            or len(past_returns.tail(TWO_YEAR_RETURNS)) < TWO_YEAR_RETURNS
            or len(past_changes.tail(TWO_YEAR_RETURNS)) < TWO_YEAR_RETURNS
        ):
            continue

        # Select the same nearest-to-spot strikes for all four configurations.
        for expiry, expiry_group in day.groupby("Expiry"):
            spot = expiry_group["Underlying Value"].median()

            selected = expiry_group.copy()
            selected["_distance"] = (selected["Strike Price"] - spot).abs()
            selected = selected.nsmallest(
                max_strikes_per_expiry, "_distance"
            )

            T = (expiry - valuation_date).days / 365.0
            if T <= 0:
                continue

            for _, option in selected.iterrows():
                S = float(option["Underlying Value"])
                K = float(option["Strike Price"])
                market_price = float(option["Close"])

                if S <= 0 or K <= 0:
                    continue

                # Black–Scholes call prices
                def bs_call(sigma):
                    if sigma <= 0:
                        return max(S * np.exp(-DIVIDEND_YIELD * T)
                                   - K * np.exp(-RISK_FREE_RATE * T), 0.0)

                    d1 = (
                        np.log(S / K)
                        + (RISK_FREE_RATE - DIVIDEND_YIELD + 0.5 * sigma**2) * T
                    ) / (sigma * np.sqrt(T))
                    d2 = d1 - sigma * np.sqrt(T)

                    return (
                        S * np.exp(-DIVIDEND_YIELD * T) * norm.cdf(d1)
                        - K * np.exp(-RISK_FREE_RATE * T) * norm.cdf(d2)
                    )

                # Bachelier call prices; sigma_n is in price units per sqrt(year).
                def bachelier_call(sigma_n):
                    forward = S * np.exp(
                        (RISK_FREE_RATE - DIVIDEND_YIELD) * T
                    )

                    if sigma_n <= 0:
                        return np.exp(-RISK_FREE_RATE * T) * max(
                            forward - K, 0.0
                        )

                    scale = sigma_n * np.sqrt(T)
                    d = (forward - K) / scale

                    return np.exp(-RISK_FREE_RATE * T) * (
                        (forward - K) * norm.cdf(d)
                        + scale * norm.pdf(d)
                    )

                configurations = {
                    "BS - 60-day": bs_call(vol_lr_60),
                    "BS - 2-year": bs_call(vol_lr_2y),
                    "Bachelier - 60-day": bachelier_call(vol_change_60),
                    "Bachelier - 2-year": bachelier_call(vol_change_2y),
                }

                for config, model_price in configurations.items():
                    records.append({
                        "Stock": stock_label,
                        "Ticker": ticker,
                        "Valuation Date": valuation_date,
                        "Month": valuation_date.to_period("M").strftime("%Y-%m"),
                        "Expiry": expiry,
                        "Spot": S,
                        "Strike": K,
                        "Market Price": market_price,
                        "Configuration": config,
                        "Model Price": model_price,
                        "Absolute Error": abs(model_price - market_price),
                        "BS Vol 60-day": vol_lr_60,
                        "BS Vol 2-year": vol_lr_2y,
                        "Bachelier Vol 60-day": vol_change_60,
                        "Bachelier Vol 2-year": vol_change_2y,
                    })

    return pd.DataFrame(records)

In [18]:
stock_files = {
    "ADANIENT": {
        "name": "Adani Enterprises",
        "ticker": "ADANIENT.NS",
        "pattern": "OPTSTK_ADANIENT_CE_*.csv"
    },
    "HINDUNILVR": {
        "name": "Hindustan Unilever",
        "ticker": "HINDUNILVR.NS",
        "pattern": "OPTSTK_HINDUNILVR_CE_*.csv"
    }
}
all_results = []
for symbol, info in stock_files.items():
    files = sorted(Path(".").glob(info["pattern"]))
    print(f"{info['name']}: found {len(files)} CSV files")

    for file in files:
        try:
            result = analyse_options_csv(
                csv_path=file,
                stock_label=info["name"],
                ticker=info["ticker"],
                max_strikes_per_expiry=3
            )

            if not result.empty:
                all_results.append(result)

        except Exception as e:
            print(f"Could not process {file.name}: {e}")

if not all_results:
    raise ValueError(
        "No results were generated. Check your CSV filenames and columns."
    )

results_all = pd.concat(all_results, ignore_index=True)

Adani Enterprises: found 36 CSV files
Hindustan Unilever: found 36 CSV files


In [19]:
print("Total pricing observations:", len(results_all))
print("Date range:",
      results_all["Valuation Date"].min(),
      "to",
      results_all["Valuation Date"].max())

# Monthly metrics
monthly_metrics = (
    results_all
    .groupby(["Stock", "Month", "Configuration"])
    .agg(
        MAE=("Absolute Error", "mean"),
        Total_Absolute_Error=("Absolute Error", "sum"),
        Total_Market_Premium=("Market Price", "sum"),
        Options=("Market Price", "size")
    )
    .reset_index()
)

monthly_metrics["WAPE (%)"] = np.where(
    monthly_metrics["Total_Market_Premium"] > 0,
    100 * monthly_metrics["Total_Absolute_Error"]
        / monthly_metrics["Total_Market_Premium"],
    np.nan
)

# Overall pooled metrics for each stock and configuration
overall_metrics = (
    results_all
    .groupby(["Stock", "Configuration"])
    .agg(
        MAE=("Absolute Error", "mean"),
        Total_Absolute_Error=("Absolute Error", "sum"),
        Total_Market_Premium=("Market Price", "sum"),
        Options=("Market Price", "size")
    )
    .reset_index()
)

overall_metrics["WAPE (%)"] = np.where(
    overall_metrics["Total_Market_Premium"] > 0,
    100 * overall_metrics["Total_Absolute_Error"]
        / overall_metrics["Total_Market_Premium"],
    np.nan
)

# Count monthly winners by MAE
mae_winners = monthly_metrics.loc[
    monthly_metrics.groupby(["Stock", "Month"])["MAE"].idxmin(),
    ["Stock", "Month", "Configuration", "MAE"]
].rename(columns={"Configuration": "MAE Winner"})

mae_win_counts = (
    mae_winners.groupby(["Stock", "MAE Winner"])
    .size()
    .rename("Months Won (MAE)")
    .reset_index()
    .rename(columns={"MAE Winner": "Configuration"})
)

# Count monthly winners by WAPE
wape_winners = monthly_metrics.dropna(subset=["WAPE (%)"]).loc[
    monthly_metrics.dropna(subset=["WAPE (%)"])
    .groupby(["Stock", "Month"])["WAPE (%)"].idxmin(),
    ["Stock", "Month", "Configuration", "WAPE (%)"]
].rename(columns={"Configuration": "WAPE Winner"})

wape_win_counts = (
    wape_winners.groupby(["Stock", "WAPE Winner"])
    .size()
    .rename("Months Won (WAPE)")
    .reset_index()
    .rename(columns={"WAPE Winner": "Configuration"})
)

win_summary = (
    overall_metrics[["Stock", "Configuration"]]
    .merge(mae_win_counts, on=["Stock", "Configuration"], how="left")
    .merge(wape_win_counts, on=["Stock", "Configuration"], how="left")
    .fillna({
        "Months Won (MAE)": 0,
        "Months Won (WAPE)": 0
    })
)

win_summary[["Months Won (MAE)", "Months Won (WAPE)"]] = (
    win_summary[["Months Won (MAE)", "Months Won (WAPE)"]].astype(int)
)

Total pricing observations: 33276
Date range: 2023-01-02 00:00:00 to 2025-12-31 00:00:00


In [20]:
print("\nOVERALL RESULTS")
display(overall_metrics[["Stock", "Configuration", "Options", "MAE", "WAPE (%)"]].round(3))
print("\nMONTHLY WIN COUNTS")
display(win_summary)
print("\nMONTHLY RESULTS")
display(monthly_metrics[["Stock", "Month", "Configuration", "Options", "MAE", "WAPE (%)"]].round(3))


OVERALL RESULTS


,Stock,Configuration,Options,MAE,WAPE (%)
0,Adani Enterprises,BS - 2-year,4089,139.738,71.808
1,Adani Enterprises,BS - 60-day,4089,152.745,78.491
2,Adani Enterprises,Bachelier - 2-year,4089,146.267,75.163
3,Adani Enterprises,Bachelier - 60-day,4089,150.270,77.220
4,Hindustan Unilever,BS - 2-year,4230,90.025,90.515
5,Hindustan Unilever,BS - 60-day,4230,88.696,89.178
6,Hindustan Unilever,Bachelier - 2-year,4230,89.795,90.284
7,Hindustan Unilever,Bachelier - 60-day,4230,88.512,88.994



MONTHLY WIN COUNTS


,Stock,Configuration,Months Won (MAE),Months Won (WAPE)
0,Adani Enterprises,BS - 2-year,13,13
1,Adani Enterprises,BS - 60-day,9,9
2,Adani Enterprises,Bachelier - 2-year,6,6
3,Adani Enterprises,Bachelier - 60-day,8,8
4,Hindustan Unilever,BS - 2-year,0,0
5,Hindustan Unilever,BS - 60-day,5,5
6,Hindustan Unilever,Bachelier - 2-year,13,13
7,Hindustan Unilever,Bachelier - 60-day,17,17



MONTHLY RESULTS


,Stock,Month,Configuration,Options,MAE,WAPE (%)
0,Adani Enterprises,2023-01,BS - 2-year,179,245.866,96.033
1,Adani Enterprises,2023-01,BS - 60-day,179,244.332,95.434
2,Adani Enterprises,2023-01,Bachelier - 2-year,179,249.028,97.268
3,Adani Enterprises,2023-01,Bachelier - 60-day,179,238.678,93.225
4,Adani Enterprises,2023-02,BS - 2-year,7,828.749,422.631
...,...,...,...,...,...,...
279,Hindustan Unilever,2025-11,Bachelier - 60-day,132,12.158,28.508
280,Hindustan Unilever,2025-12,BS - 2-year,133,18.272,48.772
281,Hindustan Unilever,2025-12,BS - 60-day,133,10.844,28.946
282,Hindustan Unilever,2025-12,Bachelier - 2-year,133,19.176,51.186


In [21]:
#pd.set_option('display.max_rows',None)
pd.reset_option('display.max_rows')

## DASHBOARD

In [22]:
import pickle
dashboard_data = {
    "results_all": results_all,
    "overall_metrics": overall_metrics,
    "monthly_metrics": monthly_metrics,
    "win_summary": win_summary
}
with open("dashboard_data.pkl", "wb") as f:
    pickle.dump(dashboard_data, f)
print("Dashboard data saved successfully!")

Dashboard data saved successfully!


In [23]:
import os
print("Folder:", os.getcwd())
print("File exists:", os.path.exists("dashboard_data.pkl"))

Folder: C:\Users\Ayush\Documents\Research Papers
File exists: True


In [24]:
import os
print(os.getcwd())

C:\Users\Ayush\Documents\Research Papers


In [30]:
import sys
!{sys.executable} -m streamlit run app.py

^C


In [32]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless", "true"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Errors:", result.stderr)

TimeoutExpired: Command '['C:\\Users\\Ayush\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'streamlit', 'run', 'app.py', '--server.headless', 'true']' timed out after 30 seconds

In [33]:
import os

print("Current folder:", os.getcwd())
print("app.py exists:", os.path.exists("app.py"))
print("Dashboard data exists:", os.path.exists("dashboard_data.pkl"))

Current folder: C:\Users\Ayush\Documents\Research Papers
app.py exists: True
Dashboard data exists: True


In [34]:
import os
import sys
import importlib.util

print("Python:", sys.executable)
print("Streamlit installed:", importlib.util.find_spec("streamlit") is not None)
print("Current folder:", os.getcwd())
print("app.py exists:", os.path.exists("app.py"))
print("Dashboard data exists:", os.path.exists("dashboard_data.pkl"))

Python: C:\Users\Ayush\AppData\Local\Python\pythoncore-3.14-64\python.exe
Streamlit installed: True
Current folder: C:\Users\Ayush\Documents\Research Papers
app.py exists: True
Dashboard data exists: True


In [35]:
!streamlit run app.py

'streamlit' is not recognized as an internal or external command,
operable program or batch file.


In [36]:
python -m venv venv

SyntaxError: invalid syntax (717537615.py, line 1)

In [37]:
sudo apt-get install python3-venv

SyntaxError: invalid syntax (1491510740.py, line 1)

In [ ]:
!python -m streamlit run app.py

In [1]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# Project folder
folder = Path(r"C:\Users\Ayush\Documents\Research Papers")

# Stocks and Yahoo Finance tickers
tickers = {
    "ADANIENT.NS": "Adani Enterprises",
    "HINDUNILVR.NS": "Hindustan Unilever"
}

all_prices = []

for ticker, stock_name in tickers.items():
    data = yf.download(
        ticker,
        start="2020-12-01",
        end="2026-01-01",
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        print(f"No data downloaded for {stock_name}")
        continue

    prices = data[["Close"]].reset_index()
    prices.columns = ["Date", "Price"]
    prices["Ticker"] = ticker
    prices["Stock"] = stock_name

    all_prices.append(prices)

# Combine both stocks into one file
stock_prices = pd.concat(all_prices, ignore_index=True)

stock_prices["Date"] = pd.to_datetime(stock_prices["Date"]).dt.tz_localize(None)

# Save CSV
csv_path = folder / "stock_price_history.csv"
stock_prices.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
print("Rows downloaded:", len(stock_prices))
print(stock_prices.head())
print(stock_prices.tail())

Saved to: C:\Users\Ayush\Documents\Research Papers\stock_price_history.csv
Rows downloaded: 2516
        Date       Price       Ticker              Stock
0 2020-12-01  420.689575  ADANIENT.NS  Adani Enterprises
1 2020-12-02  452.448212  ADANIENT.NS  Adani Enterprises
2 2020-12-03  446.265961  ADANIENT.NS  Adani Enterprises
3 2020-12-04  444.670593  ADANIENT.NS  Adani Enterprises
4 2020-12-07  456.237274  ADANIENT.NS  Adani Enterprises
           Date        Price         Ticker               Stock
2511 2025-12-24  2259.220215  HINDUNILVR.NS  Hindustan Unilever
2512 2025-12-26  2262.387939  HINDUNILVR.NS  Hindustan Unilever
2513 2025-12-29  2270.208496  HINDUNILVR.NS  Hindustan Unilever
2514 2025-12-30  2267.139648  HINDUNILVR.NS  Hindustan Unilever
2515 2025-12-31  2292.580811  HINDUNILVR.NS  Hindustan Unilever


In [2]:
import pandas as pd
from pathlib import Path

folder = Path(r"C:\Users\Ayush\Documents\Research Papers")
csv_path = folder / "stock_price_history.csv"

stock_prices = pd.read_csv(csv_path, parse_dates=["Date"])

print("Shape:", stock_prices.shape)
print("\nStocks:")
print(stock_prices["Stock"].value_counts())

print("\nDate range:")
print(stock_prices.groupby("Stock")["Date"].agg(["min", "max"]))

print("\nFirst five rows:")
print(stock_prices.head())

Shape: (2516, 4)

Stocks:
Stock
Adani Enterprises     1258
Hindustan Unilever    1258
Name: count, dtype: int64

Date range:
                          min        max
Stock                                   
Adani Enterprises  2020-12-01 2025-12-31
Hindustan Unilever 2020-12-01 2025-12-31

First five rows:
        Date       Price       Ticker              Stock
0 2020-12-01  420.689575  ADANIENT.NS  Adani Enterprises
1 2020-12-02  452.448212  ADANIENT.NS  Adani Enterprises
2 2020-12-03  446.265961  ADANIENT.NS  Adani Enterprises
3 2020-12-04  444.670593  ADANIENT.NS  Adani Enterprises
4 2020-12-07  456.237274  ADANIENT.NS  Adani Enterprises
